# EEG Splitting Strategies — Methods Criticized by the Paper

Reference: Varoquaux et al. (2016) Assessing and tuning brain decoders: cross-validation, caveats, and guidelines

This notebook implements the two strategies the paper **criticises** as biased:

| Method | Problem |
|--------|---------|
| **Leave-One-Trial-Out (LOTO)** | Trials within a subject are correlated — test set not independent from train — strongly optimistic bias |
| **Random Trial Split (non-group-aware ShuffleSplit)** | Train and test share subjects — model memorises subject-specific patterns |

Both ignore the natural block structure (subjects/sessions) and report inflated accuracy.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.model_selection import LeaveOneOut, ShuffleSplit
from sklearn.metrics import accuracy_score

import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [ ]:
DATA_DIR = Path("prepared_bnci2014_001")

X        = np.load(DATA_DIR / "X.npy", allow_pickle=True)
y_raw    = np.load(DATA_DIR / "y.npy", allow_pickle=True)
metadata = pd.read_csv(DATA_DIR / "metadata.csv")

le = LabelEncoder()
y  = le.fit_transform(y_raw)
subjects = metadata["subject"].values

print(f"X shape     : {X.shape}")
print(f"Classes     : {le.classes_}")
print(f"Subjects    : {np.unique(subjects)}")
print(f"Chance level: {1/len(le.classes_):.3f}")

In [ ]:
# Log-variance per channel (22 features). Classical EEG motor imagery feature.
X_feat = np.log(np.var(X, axis=2) + 1e-10)
print(f"Feature matrix: {X_feat.shape}")

In [ ]:
clf = Pipeline([
    ("scaler", StandardScaler()),
    ("svm",    LinearSVC(C=1.0, max_iter=2000))
])

## Method 1 (Criticised): Leave-One-Trial-Out (LOTO)

Each fold leaves out exactly **one trial**; all other 5183 trials (including thousands from the same subject) are used for training.

**Why the paper says this is wrong:** EEG trials within a subject are temporally correlated. The model trains on the subject it is tested on.

Paper (Section 3.2): leave one sample out is very optimistic in within-subject settings, as samples are highly correlated.

WARNING: LOTO over 5184 trials is slow. We run it on a stratified subsample of 500 trials.

In [ ]:
rng = np.random.default_rng(42)
n_per_class = 125
sub_idx = np.concatenate([
    rng.choice(np.where(y == c)[0], size=n_per_class, replace=False)
    for c in range(len(le.classes_))
])
sub_idx.sort()

X_sub = X_feat[sub_idx]
y_sub = y[sub_idx]
s_sub = subjects[sub_idx]

unique, counts = np.unique(s_sub, return_counts=True)
print(f"Subsample size: {len(sub_idx)} trials")
print(f"Subjects in subsample: {dict(zip(unique, counts))}")

In [ ]:
loo = LeaveOneOut()
scores_loto = []

for train_idx, test_idx in loo.split(X_sub):
    X_tr, X_te = X_sub[train_idx], X_sub[test_idx]
    y_tr, y_te = y_sub[train_idx], y_sub[test_idx]
    clf.fit(X_tr, y_tr)
    scores_loto.append(int(clf.predict(X_te)[0] == y_te[0]))

print(f"LOTO accuracy : {np.mean(scores_loto):.3f}  (n={len(scores_loto)} folds)")
print(f"Chance level  : 0.250")
print()
print("NOTE: accuracy is inflated — model trained on data from the same subjects it was tested on.")

## Method 2 (Criticised): Random Trial Split — non-group-aware ShuffleSplit

Trials are randomly shuffled and split 80/20 **ignoring which subject they come from**.

**Why the paper says this is wrong:** The model sees most of a subject data during training and is tested on the remaining trials of that same subject. EEG is highly subject-specific, so the model exploits subject identity rather than learning general motor imagery patterns.

Paper (Section 2.1): the test set should contain new samples displaying all confounding uncontrolled sources of variability — for multi-subject data, predictions on new subjects give results that hold at the population level.

In [ ]:
ss = ShuffleSplit(n_splits=10, test_size=0.2, random_state=42)
scores_rts = []

for i, (train_idx, test_idx) in enumerate(ss.split(X_feat)):
    X_tr, X_te = X_feat[train_idx], X_feat[test_idx]
    y_tr, y_te = y[train_idx],      y[test_idx]
    clf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, clf.predict(X_te))
    scores_rts.append(acc)
    leaked = sorted(set(subjects[train_idx]) & set(subjects[test_idx]))
    print(f"  Split {i+1:2d}: accuracy={acc:.3f}  | subjects in BOTH train & test (leakage): {leaked}")

print(f"\nRandom Trial Split mean +/- std : {np.mean(scores_rts):.3f} +/- {np.std(scores_rts):.3f}")
print(f"Chance level                    : 0.250")
print()
print("NOTE: ALL 9 subjects appear in both train and test in every split — data leakage.")

## Summary

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(scores_loto, bins=3, color='tomato', edgecolor='white')
ax.axvline(np.mean(scores_loto), color='darkred', linestyle='--', label=f'Mean={np.mean(scores_loto):.3f}')
ax.axvline(0.25, color='navy', linestyle=':', label='Chance=0.250')
ax.set_xlabel('Correct (1) / Wrong (0)')
ax.set_ylabel('Count')
ax.set_title('Leave-One-Trial-Out (LOTO)\n[CRITICISED]')
ax.legend()

ax = axes[1]
ax.plot(range(1, len(scores_rts)+1), scores_rts, marker='o', color='tomato', label='Split accuracy')
ax.axhline(np.mean(scores_rts), color='darkred', linestyle='--', label=f'Mean={np.mean(scores_rts):.3f}')
ax.axhline(0.25, color='navy', linestyle=':', label='Chance=0.250')
ax.set_xlabel('Split #')
ax.set_ylabel('Accuracy')
ax.set_title('Random Trial Split non-group-aware (10x, 20% test)\n[CRITICISED]')
ax.set_xticks(range(1, len(scores_rts)+1))
ax.set_ylim(0, 1)
ax.legend()

plt.tight_layout()
plt.savefig('criticised_methods.png', dpi=150)
plt.show()

print("=" * 65)
print(f"Method                         | Mean  |  Std  | Folds")
print(f"-------------------------------|-------|-------|------")
print(f"LOTO (criticised)              | {np.mean(scores_loto):.3f} | {np.std(scores_loto):.3f} | {len(scores_loto)}")
print(f"Random Trial Split (criticised)| {np.mean(scores_rts):.3f} | {np.std(scores_rts):.3f} | {len(scores_rts)}")
print(f"Chance                         | 0.250 |   -   |   -")
print("=" * 65)